to restart the stream from schema evolution  to practice


In [0]:
# Reset - Stop active streams
#Step 1 — Stop the stream

for q in spark.streams.active:
    q.stop()

print("All active streams stopped.")

All active streams stopped.


In [0]:
#Step 2 — Delete the old table and demo folder
# Reset - Remove previous schema evolution data

table_name = "retail.sales.orders_schema_evolution"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

dbutils.fs.rm(base, True)

print("Schema evolution demo reset.")

Schema evolution demo reset.


In [0]:
#Step 3 — Recreate the landing folder
# Reset - Recreate landing folder

dbutils.fs.mkdirs(landing)

print("Fresh schema evolution environment created.")

Fresh schema evolution environment created.


Schema evoltion setup process started here


In [0]:
# Step 09 - Schema evolution setup

catalog = "retail"
schema = "sales"
volume = "autoloader_demo"

base = f"/Volumes/{catalog}/{schema}/{volume}/schema_evolution"

landing = f"{base}/landing"
schema_loc = f"{base}/_schema"
checkpoint = f"{base}/_checkpoint"

dbutils.fs.mkdirs(landing)

print("Schema evolution landing folder created.")

Schema evolution landing folder created.


In [0]:
# Schema Evolution - Auto Loader
# Step 10 - Create initial JSON file


# Create initial file
dbutils.fs.put(
    f"{landing}/orders_01.json",
    '{"order_id": 1, "customer": "Aditi", "amount": 250}\n'
    '{"order_id": 2, "customer": "Rahul", "amount": 120}\n',
    True
)

print("Initial JSON file created.")



Wrote 104 bytes.
Initial JSON file created.


In [0]:
# Define Auto Loader

# Step 11 - Define Auto Loader


orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing)
)

print("Auto Loader stream defined.")



Auto Loader stream defined.


In [0]:
# Step 12 - Start initial stream

table_name = f"{catalog}.{schema}.orders_schema_evolution"

# Start stream
query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(table_name)
)

print("Initial data loaded.")


Initial data loaded.


In [0]:
# Step 13 - Check initial data

display(
    spark.sql(f"""
        SELECT *
        FROM {table_name}
        ORDER BY order_id
    """)
)



amount,customer,order_id,_rescued_data
250,Aditi,1,null
120,Rahul,2,null


In [0]:
# Check initial data
display(spark.table(table_name))

amount,customer,order_id,_rescued_data
250,Aditi,1,null
120,Rahul,2,null


In [0]:
# Check whether the table exists and has data

display(spark.sql("SHOW TABLES IN retail.sales"))

database,tableName,isTemporary
sales,customers,false
sales,customers_staged,false
sales,orders_bronze,false
sales,orders_schema_evolution,false
sales,product_updates,false
sales,stores,false
sales,transactions,false
sales,transactions_raw,false


In [0]:
display(spark.sql("""
SELECT *
FROM retail.sales.orders_schema_evolution
"""))

amount,customer,order_id,_rescued_data
250,Aditi,1,null
120,Rahul,2,null


In [0]:
# Check whether the table exists and has data

display(spark.sql("SHOW TABLES IN retail.sales"))

database,tableName,isTemporary
sales,customers,false
sales,customers_staged,false
sales,orders_bronze,false
sales,orders_schema_evolution,false
sales,product_updates,false
sales,stores,false
sales,transactions,false
sales,transactions_raw,false


In [0]:
display(dbutils.fs.ls(landing))

path,name,size,modificationTime
dbfs:/Volumes/retail/sales/autoloader_demo/schema_evolution/landing/orders_01.json,orders_01.json,104,1790492302000
dbfs:/Volumes/retail/sales/autoloader_demo/schema_evolution/landing/orders_02.json,orders_02.json,73,1790492440000


In [0]:
print(query.status)

{'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
## Step 14 - Add a new column to the source



dbutils.fs.put(
    f"{landing}/orders_02.json",
    '{"order_id": 3, "customer": "Meera", "amount": 890, "channel": "Online"}\n',
    True
)

print("New file with changed schema added.")

Wrote 73 bytes.
New file with changed schema added.


In [0]:
# Step 15 - Process the new schema

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing)
)

query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(table_name)
)

print("Schema evolution processed.")

Schema evolution processed.


In [0]:
# Step 16 - Verify the evolved table

display(
    spark.sql(f"""
        SELECT *
        FROM {table_name}
        ORDER BY order_id
    """)
)

amount,customer,order_id,_rescued_data,channel
250,Aditi,1,null,null
120,Rahul,2,null,null
890,Meera,3,null,Online


In [0]:
# Step 17 - Add a file with an unexpected data type

dbutils.fs.put(
    f"{landing}/orders_03.json",
    '{"order_id": 4, "customer": "Sanjay", "amount": "NOT_A_NUMBER", "channel": "Store"}\n',
    True
)

print("Invalid data file added.")
print("File with unexpected data type added.")

Wrote 84 bytes.
Invalid data file added.
File with unexpected data type added.


In [0]:
# Step 18 - Process the file with unexpected data type
# Step 18 - Process invalid record


orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing)
)

query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(table_name)
)

print("Unexpected data type file processed.")

Unexpected data type file processed.


In [0]:
# Step 19 - Check final data

display(
    spark.sql(f"""
        SELECT *
        FROM {table_name}
        ORDER BY order_id
    """)
)

amount,customer,order_id,_rescued_data,channel
250,Aditi,1,null,null
120,Rahul,2,null,null
890,Meera,3,null,Online
null,Sanjay,4,"{""amount"":""NOT_A_NUMBER"",""_file_path"":""/Volumes/retail/sales/autoloader_demo/schema_evolution/landing/orders_03.json""}",Store
